In [ ]:
import panel as pn
import pandas as pd
from config.database import SessionLocal
from models import Pagamento
from datetime import date


pn.extension('tabulator')

status_alert = pn.pane.Alert("", alert_type="light", visible=False)

# O campo ID fica desabilitado pois é autoincrement no banco
txt_id = pn.widgets.TextInput(name="ID Pagamento (Deixe vazio p/ novo)", disabled=False)
txt_cpf_resp = pn.widgets.TextInput(name="CPF Responsável (Chave Estrangeira)")
txt_cpf_cuid = pn.widgets.TextInput(name="CPF Cuidador (Chave Estrangeira)")
num_valor = pn.widgets.FloatInput(name="Valor (R$)", value=0.0)

btn_salvar = pn.widgets.Button(name="Lançar/Atualizar Pagamento", button_type="success")
btn_excluir = pn.widgets.Button(name="Excluir Selecionado", button_type="danger")

tabela_dados = pn.widgets.Tabulator(pd.DataFrame(), height=300, layout='fit_columns', selectable=True)

def carregar_dados():
    db = SessionLocal()
    try:
        query = db.query(Pagamento).all()
        dados = [{
            "ID": p.id_pagamento,
            "CPF Responsavel": p.cpf_responsavel,
            "CPF Cuidador": p.cpf_cuidador,
            "Valor": float(p.valor) if p.valor else 0.0,
            "Data": p.data_pagamento
        } for p in query]
        tabela_dados.value = pd.DataFrame(dados)
    finally:
        db.close()

def salvar_registro(event):
    if not txt_cpf_resp.value or not txt_cpf_cuid.value:
        status_alert.object = "CPFs do Responsável e Cuidador são obrigatórios!"
        status_alert.alert_type = "danger"
        status_alert.visible = True
        return

    db = SessionLocal()
    try:
        # Se um ID foi digitado, tenta atualizar. Senão, cria um novo.
        pagamento = None
        if txt_id.value.isdigit():
            pagamento = db.query(Pagamento).filter_by(id_pagamento=int(txt_id.value)).first()

        if not pagamento:
            pagamento = Pagamento(
                cpf_responsavel=txt_cpf_resp.value,
                cpf_cuidador=txt_cpf_cuid.value,
                valor=num_valor.value,
                data_pagamento=date.today()
            )
            db.add(pagamento)
            msg = "Pagamento lançado com sucesso!"
        else:
            pagamento.cpf_responsavel = txt_cpf_resp.value
            pagamento.cpf_cuidador = txt_cpf_cuid.value
            pagamento.valor = num_valor.value
            msg = "Pagamento atualizado com sucesso!"

        db.commit()
        status_alert.object = msg
        status_alert.alert_type = "success"
        status_alert.visible = True
        txt_id.value = "" # limpa o campo
        carregar_dados()
    except Exception as e:
        db.rollback()
        status_alert.object = f"Erro: Verifique se os CPFs existem nas outras tabelas. ({str(e)})"
        status_alert.alert_type = "danger"
        status_alert.visible = True
    finally:
        db.close()

def excluir_registro(event):
    selecao = tabela_dados.selection
    if not selecao:
        status_alert.object = "Selecione um pagamento na tabela para excluir."
        status_alert.alert_type = "warning"
        status_alert.visible = True
        return

    id_selecionado = tabela_dados.value.iloc[selecao[0]]['ID']
    db = SessionLocal()
    try:
        pagamento = db.query(Pagamento).filter_by(id_pagamento=int(id_selecionado)).first()
        if pagamento:
            db.delete(pagamento)
            db.commit()
            status_alert.object = "Pagamento cancelado/excluído com sucesso!"
            status_alert.alert_type = "success"
            status_alert.visible = True
            carregar_dados()
    except Exception as e:
        db.rollback()
        status_alert.object = f"Erro ao excluir: {str(e)}"
        status_alert.alert_type = "danger"
        status_alert.visible = True
    finally:
        db.close()

btn_salvar.on_click(salvar_registro)
btn_excluir.on_click(excluir_registro)

dashboard = pn.Column(
    "## Gestão de Pagamentos", status_alert,
    pn.Row(txt_id, txt_cpf_resp, txt_cpf_cuid), pn.Row(num_valor),
    pn.Row(btn_salvar, btn_excluir),
    "### Histórico de Pagamentos", tabela_dados
)
carregar_dados()
dashboard.servable()